# Q1 GPS Quality Assessment

This notebook identifies GPS observations with documented quality concerns. It reads from PostGIS, writes flags to a separate processed table, and does not delete or modify any record in `raw.gps_points_raw`.

## 1. Environment and reproducible paths

Run from within the Q1 project. Database credentials are supplied through environment variables; the summary output path is resolved relative to the project directory.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\umary\Desktop\EHA test\Q1_Campaign_Team_Tracking")

sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.db_connection import get_connection
from src.quality.gps_quality_rules import (
    CONFIG,
    build_flag_records,
    evaluate_gps_quality,
    quality_summary,
    upsert_quality_flags,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'Q1_Campaign_Team_Tracking':
    PROJECT_ROOT = next(path for path in (PROJECT_ROOT, *PROJECT_ROOT.parents) if path.name == 'Q1_Campaign_Team_Tracking')
OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'tables' / 'gps_quality_summary.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Quality summary output: {OUTPUT_PATH}')

In [ ]:
%pip install psycopg[binary]

## 2. Load raw GPS observations

The query reads the raw layer only. No filters are applied so missing and anomalous values remain available for flagging and review.

In [ ]:
GPS_QUERY = '''
SELECT
    gps_track_point_id, source_file, team_id, logger_id, observed_at,
    longitude, latitude, accuracy_m, speed_kmh
FROM raw.gps_points_raw
ORDER BY team_id, logger_id, observed_at, gps_track_point_id
'''

with get_connection() as connection:
    gps_points = pd.read_sql_query(GPS_QUERY, connection)

if gps_points.empty:
    raise RuntimeError('No GPS observations are available in raw.gps_points_raw. Run GPS ingestion first.')

gps_points.head()

## 3. Apply documented QA rules

Rules cover speed, reported-versus-calculated speed disagreement, positional accuracy, campaign hours, sequence gaps, and stationary clusters. Thresholds are provisional screening rules documented in `src/quality/README.md`; flags do not establish that a record is invalid.

In [ ]:
qa_points = evaluate_gps_quality(gps_points, CONFIG)
flag_records = build_flag_records(qa_points, CONFIG)
summary = quality_summary(flag_records, len(qa_points))
summary

## 4. Store flags and export the summary

Only the processed flag table is written. The upsert key is raw-point reference plus quality rule, so re-running the assessment does not create duplicate flags.

In [ ]:
with get_connection() as connection:
    upsert_quality_flags(connection, flag_records)

summary.to_csv(OUTPUT_PATH, index=False)
print(f'Flag records written: {len(flag_records):,}')
print(f'Summary written: {OUTPUT_PATH}')

## 5. Diagnostic visualizations

These diagnostic plots support review of flagged observations. They are not final cartographic products and do not make settlement-attribution or hotspot decisions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(qa_points['calculated_speed_kmh'].dropna(), bins=60, color='#4472C4')
axes[0, 0].axvline(CONFIG.maximum_plausible_speed_kmh, color='#C00000', linestyle='--')
axes[0, 0].set(title='Calculated speed distribution', xlabel='km/h', ylabel='GPS points')

axes[0, 1].hist(qa_points['accuracy_m'].dropna(), bins=60, color='#70AD47')
axes[0, 1].axvline(CONFIG.maximum_acceptable_accuracy_m, color='#C00000', linestyle='--')
axes[0, 1].set(title='Reported positional accuracy', xlabel='Accuracy (m)', ylabel='GPS points')

axes[1, 0].hist(qa_points['sequence_gap_minutes'].dropna(), bins=60, color='#ED7D31')
axes[1, 0].axvline(CONFIG.sequence_gap_minutes, color='#C00000', linestyle='--')
axes[1, 0].set(title='Sequential time gaps', xlabel='Minutes', ylabel='GPS points')

flagged_points = qa_points[qa_points[['impossible_speed_flag', 'accuracy_quality_flag', 'outside_campaign_hours_flag', 'gps_gap_flag', 'stationary_cluster_flag']].any(axis=1)]
axes[1, 1].scatter(qa_points['longitude'], qa_points['latitude'], s=1, alpha=0.15, label='All points')
axes[1, 1].scatter(flagged_points['longitude'], flagged_points['latitude'], s=3, alpha=0.6, color='#C00000', label='Flagged points')
axes[1, 1].set(title='Flagged-point diagnostic view', xlabel='Longitude', ylabel='Latitude')
axes[1, 1].legend(markerscale=3)

fig.tight_layout()

## 6. Interpretation boundary

The output of this phase is **flagged GPS observations with documented quality concerns**. Flagged observations require review in later processing; this notebook does not delete points, choose settlement-attribution tolerances, run hotspot analysis, or create final maps.